# 第2章 卷积

本章节学习目标：

- 理解本章核心算法的**数学原理**
- 掌握算法的**手写实现**方法
- 学会使用 OpenCV 对应函数进行**工程实践**
- 通过编程练习加深对算法的理解

> **📌 学习建议**：先阅读概念说明，再动手编写代码，最后完成练习


In [ ]:
# -*- coding: utf-8 -*-
# 中文路径兼容的图像读写函数
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False


# 代码实现

我们动手编写一维卷积函数。

In [ ]:
import matplotlib.pyplot as plt


# 一维卷积
class conv_1d():
    def __init__(self, a, b):
        # 输入信号
        self.a = a
        # 卷积核
        self.b = b
        # 输入信号的坐标，默认从0开始。
        self.ax = [i for i in range(len(a))]
        # 卷积核的坐标，默认从0开始。
        self.bx = [i for i in range(len(b))]

    def conv(self):
        lst1 = self.a
        lst2 = self.b
        l1 = len(lst1)
        l2 = len(lst2)
        lst1 = [0] * (l2 - 1) + lst1 + [0] * (l2 - 1)
        lst2.reverse()
        c = [0 for x in range(0, l1 + l2 - 1)]
        for i in range(l1 + l2 - 1):
            for j in range(l2):
                c[i] += lst1[i + j] * lst2[j]
        return c

    def plot(self):
        a = self.a
        b = self.b
        ax = self.ax
        bx = self.bx

        # 为了更直观的查看结果，我们分别绘制a、b与它们卷积得到的信号
        plt.figure(1)
        # 图一包含1行2列子图，当前画在第一行第一列图上
        plt.subplot(1, 2, 1)
        plt.title('Input')
        plt.bar(ax, a, color='lightcoral', width=0.2)
        plt.plot(ax, a, color='lightcoral')

        plt.figure(1)
        # 当前画在第一行第2列图上
        plt.subplot(1, 2, 2) 
        plt.title('Kernel')
        plt.bar(bx, b, color='lightgreen', width=0.2)
        #plt.plot(bx, b, color='lightgreen')


        # 计算输出信号以及其坐标，并画图 
        c = self.conv()
        length = len(c)
        cx = [i for i in range(length)]
        plt.figure()
        plt.title('Output')
        plt.bar(cx, c, color='lightseagreen', width=0.2)
        plt.plot(cx, c, color='lightseagreen')

现在我们举例来显示上述卷积实现的效果。用不同的卷积核去对一个三角波信号进行卷积，然后观察卷积的效果。

In [ ]:
# 定义输入信号与卷积核
a = [0,1,2,3,2,1,0]

# 冲激函数
k = [0,0,1,0,0]

conv = conv_1d(a, k)
conv.plot()

当所用的卷积核是一个单位冲激信号（即“面积”等于1只在一个位置出现的窄脉冲）时，输入信号在卷积之后并没有发生变化。这是卷积的一个重要性质，请大家记住。

下面更换一下卷积核，看看输出会有什么不同的效果。

In [ ]:
# 定义输入信号与卷积核
a = [0,1,2,3,2,1,0]
# 方波信号
k = [1,1,1,1,1]

conv = conv_1d(a, k)
conv.plot()

大家是否注意到，用一个方波信号对一个三角波信号进行卷积，其效果是将三角波变得平滑了。请大家记住这个现象，后续会再次提到。


接下来，我们开始编写二维卷积。我们将对一张图像进行处理，并直观的展示卷积前后图像的变化。和一维卷积类似，也使用冲激信号和方波信号作为卷积核。大家注意代码中二维冲激信号和二维方波信号的实现。这里直接调用Python中的库函数cv2.filter2D()进行二维卷积操作。我们把二维卷积的代码实现作为留给大家的习题，请大家自己完成。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import seaborn as sns

class conv_2d():
    def __init__(self, image, kernel):
        self.img = image
        self.k = kernel
        
    def plot(self):
        # 展示输入图像
        plt.imshow(self.img[:, :, ::-1])
        plt.axis('off')
        plt.title('Input')
        plt.show()
    
        # 展示卷积核
        fig = plt.figure(figsize=(2, 1.5))
        sns.heatmap(self.k)
        plt.axis('off')
        plt.title('Kernel')
        
    # 定义二维卷积
    def convolution(self, data, k):
        # 直接调用库函数进行卷积操作
        return cv2.filter2D(data, -1, k)

    
    # 展示二维卷积结果
    def plot_conv(self):
        # 卷积过程
        img_new = self.convolution(self.img, self.k)
        # 卷积结果可视化            
        plt.figure()
        plt.imshow(img_new[:, :, ::-1])
        plt.title('Output')
        plt.axis('off')
        return


为了后续章节方便使用，我们编写utils.py文件并将上述代码导入该文件中。

我们先来观察一下使用冲击信号对图像卷积的结果。

In [ ]:
img =cv2.imread('lena.jpeg')

# 二维冲激卷积核
size = 15
k1 = np.zeros((size, size))
mid = (size-1) // 2
k1[mid][mid] = 1

# 展示输入图像与卷积核
conv = conv_2d(img, k1)
conv.plot()
\
# 展示卷积结果
conv.plot_conv()


由结果可见，用二维冲激函数对图像进行卷积，得到的结果和原图是一样的。这与进行一维卷积得到的结果一致。

接着，我们再来观察使用方波信号对图像卷积的结果。

In [ ]:
# 二维方波卷积核
size = 15
# 因为二维的核函数的大小是n*n的，因此在实现方波信号时我们需要除以(size*size)
k2 = 1/size/size * np.ones((size, size))

# 展示输入图像与卷积核
conv = conv_2d(img, k2)
conv.plot()

# 展示卷积结果
conv.plot_conv()

由结果可见，用方波信号作为卷积核，对图像做卷积，得到的结果是平滑模糊的图像，这也与我们在一维卷积中的实验结果一致。这是不是有一些图像处理的味道了？请大家记住这个结果，我们将在之后的章节中进一步阐述其原理。


---

## 📝 练习：手写卷积操作


**练习目标**：不使用 cv2.filter2D 等函数，手写实现二维卷积操作。

**要求**：
1. 实现 `convolve2d_manual(image, kernel)` 函数
2. 支持任意大小的卷积核
3. 处理边界填充问题（zero padding / same padding）
4. 用手写卷积和 cv2.filter2D 的结果进行对比验证


**💡 小提示**：
- 除 `cv_imread` / `cv_imwrite` 外，不直接调用 OpenCV 高层函数
- 使用 NumPy 进行矩阵运算
- 注意边界处理和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
```python
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

def convolve2d_manual(image, kernel, padding='same'):
    """
    手写二维卷积实现
    - image: 输入图像 (H, W) 或 (H, W, C)
    - kernel: 卷积核 (kH, kW)
    - padding: 'same' 输出与输入同尺寸, 'valid' 无填充
    """
    # 获取图像和卷积核尺寸
    if len(image.shape) == 2:
        image = image[:, :, np.newaxis]
    
    h, w, c = image.shape
    kh, kw = kernel.shape
    
    # 计算填充大小
    if padding == 'same':
        pad_h = kh // 2
        pad_w = kw // 2
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='constant')
        out_h, out_w = h, w
    else:  # valid
        padded = image
        out_h = h - kh + 1
        out_w = w - kw + 1
    
    # 翻转卷积核（真正的卷积操作）
    kernel_flipped = kernel[::-1, ::-1]
    
    # 执行卷积
    output = np.zeros((out_h, out_w, c), dtype=np.float32)
    for i in range(out_h):
        for j in range(out_w):
            region = padded[i:i+kh, j:j+kw, :]
            for ch in range(c):
                output[i, j, ch] = np.sum(region[:, :, ch] * kernel_flipped)
    
    # 恢复原始维度
    if len(image.shape) == 2:
        output = output[:, :, 0]
    
    return output.astype(np.uint8)

# 验证
img = cv_imread('lena.jpeg', cv2.IMREAD_GRAYSCALE).astype(np.float32)

# 均值滤波核
kernel = np.ones((5, 5), dtype=np.float32) / 25.0

# 手写卷积结果
result_manual = convolve2d_manual(img, kernel, 'same')

# OpenCV 参考结果
result_cv = cv2.filter2D(img, -1, kernel)

# 对比
diff = np.abs(result_manual.astype(float) - result_cv.astype(float))
print(f"最大差异: {diff.max():.1f}")
print(f"平均差异: {diff.mean():.1f}")
print(f"结果匹配: {np.allclose(result_manual, result_cv, atol=1)}")
```



### 💻 代码要点解释

1. **图像读取与保存**：使用自定义的 `cv_imread` / `cv_imwrite` 函数，解决 Windows 中文路径下 OpenCV 读写图像失败的问题

2. **算法核心**：手写实现的核心在于**不依赖现成库函数**，而是直接操作像素和矩阵运算

3. **对比验证**：通过与 OpenCV 对应函数的结果进行数值对比，验证手写实现的正确性

4. **参数分析**：调整算法参数，观察输出变化，理解每个参数的物理含义

5. **扩展思考**：尝试将算法应用到自己的图像上，或改进算法（如增加加速技巧）

---

</details>

---
